# DAY 1

#### EXTRACTION AND BASIC

In [1]:
from pathlib import Path
import pdfplumber as pdfloader
print(f"Current Working Directory: {Path.cwd()}")

Current Working Directory: D:\Career & Interview\RAG Learning\4.Similarity_Search\notebook


In [2]:
base_dir = Path.cwd().parent.parent  # repo root

# 2. Build the path dynamically
pdf_path = base_dir / "data" / "building" / "Muhammad-pages.pdf"

# 3. THE VITAL STEP: Check if it actually exists
if pdf_path.exists():
    print(f"Success! The file is at: {pdf_path}")
else:
    print(f"Error: No file found at {pdf_path}")
    print(f"Double-check your folder structure!")

Success! The file is at: D:\Career & Interview\RAG Learning\4.Similarity_Search\data\building\Muhammad-pages.pdf


In [3]:
wholeText= ""
with pdfloader.open(pdf_path) as pdfFile:
    for index,page in enumerate(pdfFile.pages):
        # print("Page <",index+1,"> \n",page.extract_text()) 
        wholeText += page.extract_text()
        wholeText+=" "
        if(page.extract_text()==""): print("Page was empty")
        # print("--------------------------------------------------------------")

### Step 6

#### Write code to inspect the extracted text.

#### Things to check:

#### Total number of pages.
#### Total number of characters.
#### Total number of words.
#### Whether any pages are empty.


In [4]:
print(len(wholeText))

7529


In [5]:
print(len(wholeText.split()))

1232


In [6]:
paragraphs = wholeText.split("\n")

In [ ]:
for para in paragraphs[:10]: 
    print(para)

# DAY 2
#### Chunking

In [7]:
words = wholeText.split()

### MY WAY OF DOING THE THING

In [8]:
# chunks = [] # list of chunks,we will make later
# maxChunkLength = 400
# currentText=""
# i=0
# while(i<len(words)):
#     currentText += words[i] + " "
#     if(i== maxChunkLength):
#         chunks.append(currentText)
#         currentText=""
#         i-=100
#         maxChunkLength = i + maxChunkLength
#     i+=1
# if currentText: chunks.append(currentText)
    

### Claude way of doing the same thing

In [9]:
chunks = []
maxChunkLength = 400
overlap = 100
i = 0
while i < len(words):
    chunk = words[i:i+maxChunkLength]
    chunks.append(" ".join(chunk))
    i += (maxChunkLength - overlap)

In [10]:
len(words)

1232

In [11]:
len(chunks)

5

In [12]:
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()

--- Chunk 0 ---
Abu Talib politely dismissed them at first, thinking it was just a heated talk. But as Muhammad grew more vocal, Abu Talib requested Muhammad to not burden him beyond what he could bear, to which Muhammad wept and replied that he would not stop even if they put the sun in his right hand and the moon in his left. When he turned around, Abu Talib called him and said, "Come back nephew, say what you please, for by God I will never give you up on any account."[126][127] Quraysh delegation to Yathrib The leaders of the Quraysh sent Nadr ibn al-Harith and Uqba ibn Abi Mu'ayt to Yathrib to seek the opinions of the Jewish rabbis regarding Muhammad. The rabbis advised them to ask Muhammad three questions: recount the tale of young men who ventured forth in the first age; narrate the story of a traveler who reached both the eastern and western ends of the earth; and provide details about the spirit. If Muhammad answered correctly, they stated, he would be a Prophet; otherwise, he

# DAY 3 EMBEDDINGS


#### Loading model 

In [13]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

C:\Users\muham\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 298.47it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## EMBEDDINGS

In [14]:
embeddings = model.encode(chunks)

In [15]:
len(embeddings)

5

In [16]:
embeddings

array([[ 0.00812211,  0.14528619,  0.00533355, ..., -0.02064898,
        -0.02102001, -0.04554903],
       [-0.00639197,  0.08161156, -0.08009648, ..., -0.04799549,
        -0.06368846, -0.03586169],
       [ 0.01379712,  0.08565222, -0.06468876, ..., -0.0507125 ,
        -0.00806763, -0.04196059],
       [-0.02984395,  0.16627711, -0.00402122, ..., -0.04163327,
        -0.04485998, -0.04698499],
       [-0.01489365,  0.11668813, -0.00070899, ...,  0.04949471,
        -0.01757102, -0.0540792 ]], shape=(5, 384), dtype=float32)

# DAY 4
### ASKING A QUERY FROM MODEL

In [19]:
# 02:08 pm ended session,

In [29]:
# using scikit learn

from sklearn.metrics.pairwise import cosine_similarity

query = "Why did Muhammad go to Ta'if and what happened there?"
def AskQuestion(query):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)
    listOfTuples = []
    for index,score in enumerate(similarities[0]):
        listOfTuples.append((index,score))
    # sort and get top 3
    listOfTuples.sort(key=lambda x: x[1], reverse=True)
    top_chunks = [chunks[listOfTuples[i][0]] for i in range(3)]
    top_indices = [listOfTuples[i][0] for i in range(3)]
    print(top_indices)
    return top_chunks

top_chunks = AskQuestion(query)
for i, chunk in enumerate(top_chunks):
    print(f"Result {i+1}:")
    print(chunk)
    print("-" * 50)
    

[3, 4, 0]
Result 1:
with a response: "If you are truly a prophet, what need do you have of our help? If God sent you as his messenger, why doesn't He protect you? And if Allah wished to send a prophet, couldn't He have found a better person than you, a weak and fatherless orphan?"[150] Realizing his efforts were in vain, Muhammad asked the people of Ta'if to keep the matter a secret, fearing that this would embolden the hostility of the Quraysh against him. However, instead of accepting his request, they pelted him with stones, injuring his limbs.[151] He eventually evaded this chaos and persecution by escaping to the garden of Utbah ibn Rabi'ah, a Meccan chief with a summer residence in Ta'if. Muhammad felt despair due to the unexpected rejection and hostility he received in the city; at this point, he realized he had no security or protection except from God, so he began praying. Shortly thereafter, Utbah's Christian slave Addas stopped by and offered grapes, which Muhammad accepted.

In [1]:
# pip install faiss-cpu

# DAY 5
## FAISS IMPLEMENTATION

In [23]:
import faiss as fs
# how to get dimensions from embeddings

In [24]:
embeddings.shape

(5, 384)

In [26]:
index = fs.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

### Now we don't use that cosine similarity search anymore,but a rather effecient faiss similarity engine.

In [27]:
query_2 = "Why did Muhammad go to Ta'if and what happened there?"
query_embeddings = model.encode([query_2])
index.search(query_embeddings,3)

(array([[0.81670964, 0.96936214, 1.0197638 ]], dtype=float32),
 array([[3, 4, 0]]))

# Day 6
### Retrieval Pipeline